# Figure 1 — MNI slices, normative chart, and Z-score / GOF cortical maps

- **SubA**: MNI template brain slices used as a background for the normative chart.
- **SubB**: GPR normative chart for the right caudal anterior cingulate cortex (rh-cACC), showing the model prediction (mean ± 95% CI) and the observed TDC / ASD Z-scores.
- **SubD**: Cortical surface maps of mean GrayVol Z-scores (ASD vs TDC) and epicenter GOF values.

## Panel A — MNI152 T1 template (z=0 axial slice)

In [ ]:
# Import Required Libraries
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm
from nilearn import plotting

# Output directory
output_dir = 'Fig1'
os.makedirs(output_dir, exist_ok=True)

# Use non-interactive backend
matplotlib.use('Agg')

# Global font settings -> Times New Roman
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'stix'
print(f"Output directory: {output_dir}/")

# Load and plot MNI152 T1 2mm template -- clean z=0 slice
mni_path = '/path/to/MNI152_T1_2mm.nii.gz'   # <-- set local MNI template path

fig = plotting.plot_anat(mni_path,
                         display_mode='z',
                         cut_coords=[0],
                         annotate=False,
                         draw_cross=False,
                         colorbar=False,
                         black_bg=False,
                         dim=-0.5,
                         figure=plt.figure(figsize=(6, 6)))

# Remove all axes and white border
ax = plt.gca()
ax.axis('off')
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

fig.savefig(f'{output_dir}/MNI_z0_clean.png',
            dpi=200, bbox_inches='tight', pad_inches=0, transparent=False)
plt.close()
print(f"Saved: {output_dir}/SubA_MNI_z0_clean.png")

## Panel B2 — Individual MIND connectivity heatmap (one ASD subject, CABIC)

In [ ]:
"""Panel B2: individual MIND connectivity heatmap for one ASD subject (CABIC)."""
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = 'Fig1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load subject info and pick the first ASD subject
INFO_PATH = 'data/CABIC_AGE.xlsx'
df_info = pd.read_excel(INFO_PATH)
df_asd = df_info[df_info['GROUP'] == 'ASD'].dropna(subset=['aparc']).reset_index(drop=True)
subject = df_asd.iloc[0]
subid = subject['SUBID']
mind_path = subject['aparc']
print(f"Selected subject: {subid}")
print(f"MIND file: {mind_path}")

# 2. Read the MIND connectivity matrix
df_mind = pd.read_csv(mind_path)
region_names = df_mind.columns.tolist()
n_regions = len(region_names)
matrix = df_mind.values
print(f"Matrix shape: {matrix.shape} ({n_regions}x{n_regions})")

# 3. Plot the heatmap
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(matrix, cmap='Reds', aspect='equal')
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
output_path = os.path.join(OUTPUT_DIR, 'SubB_2.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {output_path}")

## Panel C — Normative modeling percentile curves (CABIC, rh-cACC)

In [ ]:
"""Panel C: GPR normative-model percentile curves for rh-cACC (CABIC cohort).
ComBat harmonization -> Euler regression -> GPR fit on TDC -> percentile
prediction across age -> plot percentile bands with per-site scatter."""
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from neuroCombat import neuroCombat
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# --- Configuration ---
MORPHO_FEATURES_PATH = 'data/CABIC_all_aparc_features.csv'
SUBJECT_INFO_PATH = 'data/CABIC_AGE.xlsx'
EULER_COL = 'mean_euler_bh'
MORPHO_FEATURES = ['GrayVol']

# Prediction age axis (3 to 13 years, per the CABIC age range)
AGE_MIN, AGE_MAX, AGE_STEP = 3, 13, 0.1
age_grid = np.arange(AGE_MIN, AGE_MAX + AGE_STEP, AGE_STEP)

OUT_PNG = 'Fig1/SubC.png'
OUT_CSV = 'Fig1/NormativeChart_rhcACC_Trajectory.csv'

TARGET_REGION = 'rh_caudalanteriorcingulate'
PERCENTILES = {'P2_5': -1.96, 'P10': -1.28, 'P25': -0.675, 'P50': 0.0,
               'P75': 0.675, 'P90': 1.28, 'P97_5': 1.96}

# Academic plot style
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.edgecolor': 'black', 'axes.linewidth': 0.8,
                     'axes.grid': False, 'font.family': 'serif',
                     'xtick.color': 'black', 'ytick.color': 'black',
                     'xtick.direction': 'in', 'ytick.direction': 'in',
                     'xtick.labelsize': 16, 'ytick.labelsize': 16})


def plot_normative_rhcACC():
    # --- Step 1: load and align data ---
    print("--- Loading data ---")
    df_morpho = pd.read_csv(MORPHO_FEATURES_PATH)
    df_info = pd.read_excel(SUBJECT_INFO_PATH)
    df_morpho['participant_id'] = df_morpho['participant_id'].apply(lambda x: str(x).split('-')[-1])
    df_info['SUBID'] = df_info['SUBID'].astype(str)
    needed_cols = ['SUBID', 'GROUP', 'AGE', 'SEX', 'SITE', EULER_COL]
    df_all = pd.merge(df_morpho, df_info[needed_cols],
                      left_on='participant_id', right_on='SUBID', how='inner')
    df_all['SEX_code'] = df_all['SEX'].apply(lambda x: 0 if x in ['M', 'MALE'] else 1)
    df_all.dropna(subset=['AGE', 'SEX_code', 'SITE', EULER_COL], inplace=True)
    all_regions = sorted(df_all['Label'].unique())
    if TARGET_REGION not in all_regions:
        raise ValueError(f"Region {TARGET_REGION} not found in data labels!")

    # --- Step 2: reshape feature matrix + ComBat harmonization ---
    print("--- ComBat site harmonization ---")
    n_regions = len(all_regions)
    valid_ids = df_all['SUBID'].value_counts()
    valid_ids = valid_ids[valid_ids == n_regions].index.tolist()
    df_valid = df_all[df_all['SUBID'].isin(valid_ids)].copy()
    all_subjects_features, combat_ids = [], []
    for sub_id in valid_ids:
        df_sub = df_valid[df_valid['SUBID'] == sub_id].sort_values(by='Label')
        all_subjects_features.append(df_sub.set_index('Label')[MORPHO_FEATURES].values.flatten())
        combat_ids.append(sub_id)
    X_input = np.array(all_subjects_features).T
    df_covars_full = df_valid.drop_duplicates(subset=['SUBID']).set_index('SUBID').loc[combat_ids].copy()
    covars_combat = pd.DataFrame({'batch': df_covars_full['SITE'].values,
                                  'age': df_covars_full['AGE'].values,
                                  'sex': df_covars_full['SEX_code'].values,
                                  'group': df_covars_full['GROUP'].values})
    X_harmonized = neuroCombat(dat=X_input, covars=covars_combat, batch_col='batch',
                               categorical_cols=['sex', 'group'], continuous_cols=['age'])['data'].T

    # --- Step 3: explicit regression against the QC (Euler) metric ---
    print(f"--- Regressing out {EULER_COL} ---")
    euler_values = df_covars_full[EULER_COL].values.reshape(-1, 1)
    X_residualized = np.zeros_like(X_harmonized)
    for i in range(X_harmonized.shape[1]):
        y_signal = X_harmonized[:, i].reshape(-1, 1)
        reg = LinearRegression().fit(euler_values, y_signal)
        X_residualized[:, i] = (y_signal - reg.predict(euler_values)).flatten() + np.mean(y_signal)
    all_feature_names = [f"{r}_{f}" for r in all_regions for f in MORPHO_FEATURES]
    df_final = pd.DataFrame(X_residualized, columns=all_feature_names)
    df_final['SUBID'] = combat_ids
    df_final = pd.merge(df_covars_full.reset_index()[['SUBID', 'GROUP', 'AGE', 'SEX_code', 'SITE']],
                        df_final, on='SUBID')

    # --- Step 4: GPR fit on TDC -> percentile prediction ---
    df_tdc = df_final[df_final['GROUP'] == 'TDC']
    print(f"TDC sample size: {len(df_tdc)}")
    X_td = df_tdc[['AGE', 'SEX_code']].values
    scaler_X = StandardScaler().fit(X_td)
    X_td_scaled = scaler_X.transform(X_td)
    target_col = f"{TARGET_REGION}_GrayVol"
    Y_td = df_tdc[target_col].values
    y_mean, y_std = float(Y_td.mean()), float(Y_td.std())
    Y_td_norm = (Y_td - y_mean) / y_std
    kernel = C(1.0) * RBF(length_scale=[1.0, 1.0]) + WhiteKernel(noise_level=0.5)
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=42, normalize_y=False)
    gpr.fit(X_td_scaled, Y_td_norm)
    X_pred = np.column_stack([age_grid, np.zeros_like(age_grid)])
    mu_norm, sigma_norm = gpr.predict(scaler_X.transform(X_pred), return_std=True)
    mu_real = mu_norm * y_std + y_mean
    sigma_real = sigma_norm * y_std

    # Save percentile trajectory
    df_chart = pd.DataFrame({'Age': np.round(age_grid, 1)})
    for label, z in PERCENTILES.items():
        df_chart[label] = mu_real + z * sigma_real
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    df_chart.to_csv(OUT_CSV, index=False)
    print(f"Saved: {OUT_CSV} ({len(df_chart)} rows)")

    # --- Step 5: plot ---
    fig, ax = plt.subplots(figsize=(8.5, 5.0), dpi=220)
    for label, z in PERCENTILES.items():
        if label == 'P50':
            ax.plot(age_grid, df_chart[label], linestyle='-', color='#1A1A1A', linewidth=1.8)
        else:
            ax.plot(age_grid, df_chart[label], linestyle='--', color='#404040', linewidth=0.8)
    unique_sites = df_tdc['SITE'].unique()
    colors = plt.cm.get_cmap('viridis', len(unique_sites))(np.linspace(0, 0.85, len(unique_sites)))
    for idx, site in enumerate(unique_sites):
        df_site = df_tdc[df_tdc['SITE'] == site]
        ax.scatter(df_site['AGE'], df_site[target_col], s=15, color=colors[idx],
                   alpha=0.6, edgecolors='none', label=site)
    ax.set_xlabel('Age (year)', fontsize=18, labelpad=8)
    ax.set_ylabel('Volume (mm³)', fontsize=18, labelpad=8)
    ax.set_xlim(AGE_MIN, AGE_MAX)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.tick_params(top=True, right=True, which='both')
    ax.legend(loc='upper right', ncol=3, frameon=True, facecolor='white',
              edgecolor='none', fontsize=12, columnspacing=0.8, handletextpad=0.2)
    plt.tight_layout()
    os.makedirs(os.path.dirname(OUT_PNG), exist_ok=True)
    plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {OUT_PNG}")


if __name__ == '__main__':
    plot_normative_rhcACC()

In [ ]:
"""Panel D: epicenter schematic for one ASD subject.
Cortical maps of (1) Z-score deviation, (2-4) TDC-average MIND fingerprints
seeded from rh-cACC / rh-TemporalPole / rh-Cuneus, and (5) the GOF scores,
composed into a single figure."""
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from enigmatoolbox.utils.parcellation import parcel_to_surface
from enigmatoolbox.plotting import plot_cortical
from IPython.display import display, Image as IPyImage
from PIL import Image

os.environ.setdefault('DISPLAY', ':99')
try:
    proc = subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x1024x24'],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    proc.poll()
except Exception:
    pass

OUTPUT_DIR = 'Fig1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

dk68_labels = [
    'lh_bankssts', 'lh_caudalanteriorcingulate', 'lh_caudalmiddlefrontal', 'lh_cuneus',
    'lh_entorhinal', 'lh_fusiform', 'lh_inferiorparietal', 'lh_inferiortemporal',
    'lh_isthmuscingulate', 'lh_lateraloccipital', 'lh_lateralorbitofrontal', 'lh_lingual',
    'lh_medialorbitofrontal', 'lh_middletemporal', 'lh_parahippocampal', 'lh_paracentral',
    'lh_parsopercularis', 'lh_parsorbitalis', 'lh_parstriangularis', 'lh_pericalcarine',
    'lh_postcentral', 'lh_posteriorcingulate', 'lh_precentral', 'lh_precuneus',
    'lh_rostralanteriorcingulate', 'lh_rostralmiddlefrontal', 'lh_superiorfrontal',
    'lh_superiorparietal', 'lh_superiortemporal', 'lh_supramarginal', 'lh_frontalpole',
    'lh_temporalpole', 'lh_transversetemporal', 'lh_insula',
    'rh_bankssts', 'rh_caudalanteriorcingulate', 'rh_caudalmiddlefrontal', 'rh_cuneus',
    'rh_entorhinal', 'rh_fusiform', 'rh_inferiorparietal', 'rh_inferiortemporal',
    'rh_isthmuscingulate', 'rh_lateraloccipital', 'rh_lateralorbitofrontal', 'rh_lingual',
    'rh_medialorbitofrontal', 'rh_middletemporal', 'rh_parahippocampal', 'rh_paracentral',
    'rh_parsopercularis', 'rh_parsorbitalis', 'rh_parstriangularis', 'rh_pericalcarine',
    'rh_postcentral', 'rh_posteriorcingulate', 'rh_precentral', 'rh_precuneus',
    'rh_rostralanteriorcingulate', 'rh_rostralmiddlefrontal', 'rh_superiorfrontal',
    'rh_superiorparietal', 'rh_superiortemporal', 'rh_supramarginal', 'rh_frontalpole',
    'rh_temporalpole', 'rh_transversetemporal', 'rh_insula'
]

# Load the first subject's Z-scores
df_z = pd.read_csv('output/ABIDE2_Morpho_Zscore_aparc.csv')
subject_id = df_z.iloc[0]['SUBID']
print(f"Selected subject: {subject_id}")
z_values_68 = np.zeros(68)
for i, label in enumerate(dk68_labels):
    col_name = f'Zscore_{label}_GrayVol'
    if col_name in df_z.columns:
        z_values_68[i] = df_z.iloc[0][col_name]
z_abs_max = max(abs(z_values_68.min()), abs(z_values_68.max()))

# TDC-average MIND matrix
df_mind = pd.read_csv('output/TDC_Average_MIND_ABIDE2_aparc.csv')
mind_matrix = df_mind.values

# GOF scores for this subject
df_gof = pd.read_csv('output/Center/ASD_GOF_Scores_aparc.csv')
gof_row = df_gof[df_gof['SUBID'] == subject_id]
gof_values_68 = np.zeros(68)
for i, label in enumerate(dk68_labels):
    if label in gof_row.columns:
        gof_values_68[i] = gof_row[label].values[0]
gof_abs_max = max(abs(gof_values_68.min()), abs(gof_values_68.max()))

seeds = ['rh_caudalanteriorcingulate', 'rh_temporalpole', 'rh_cuneus']

# Diverging blue-white-red colormap
cmap_div_name = 'diverging_rdbu'
colors_div = ['#2166ac', '#f7f7f7', '#b2182b']
cmap_div = LinearSegmentedColormap.from_list(cmap_div_name, colors_div)
try:
    plt.colormaps.register(cmap=cmap_div, name=cmap_div_name, force=True)
except Exception:
    pass

# High-contrast non-red-blue colormap for the Z-score map
cmap_atrophy_name = 'atrophy_plasma_distinct'
colors_atrophy = ['#0d0887', '#cc4778', '#f0f0f0', '#f89c41', '#f0f921']
cmap_atrophy = LinearSegmentedColormap.from_list(cmap_atrophy_name, colors_atrophy)
try:
    plt.colormaps.register(cmap=cmap_atrophy, name=cmap_atrophy_name, force=True)
except Exception:
    pass


def extract_right_medial(full_img):
    h, w = full_img.shape[:2]
    view_w = w // 4
    return full_img[:, 2*view_w:3*view_w]


def rearrange_to_2x2(full_img):
    h, w = full_img.shape[:2]
    view_w = w // 4
    lh_lat, lh_med = full_img[:, 0:view_w], full_img[:, view_w:2*view_w]
    rh_med, rh_lat = full_img[:, 2*view_w:3*view_w], full_img[:, 3*view_w:4*view_w]
    top_row = np.concatenate([lh_lat, rh_lat], axis=1)
    bottom_row = np.concatenate([lh_med, rh_med], axis=1)
    return np.concatenate([top_row, bottom_row], axis=0)


def save_single_view(tmp_path, out_path, vals, cmap, vmin, vmax, keep_all_views=False):
    fsa5 = parcel_to_surface(vals, 'aparc_fsa5')
    plot_cortical(array_name=fsa5, surface_name='fsa5', size=(1200, 300),
                  cmap=cmap, color_bar=False, color_range=(vmin, vmax),
                  screenshot=True, filename=tmp_path, background=(1, 1, 1), scale=(3, 3))
    img_full = plt.imread(tmp_path)
    if keep_all_views:
        img_processed = rearrange_to_2x2(img_full)[:, :, :3]
        figsize = (6, 6)
    else:
        img_processed = extract_right_medial(img_full)[:, :, :3]
        figsize = (4, 4)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img_processed)
    ax.axis('off')
    plt.savefig(out_path, bbox_inches='tight', dpi=200, pad_inches=0)
    plt.close()
    return img_processed


sub_files, tmp_files = [], []

# Fig 1: Z-score map with the high-contrast colormap
tmp1 = os.path.join(OUTPUT_DIR, '_tmp_1.png')
out1 = os.path.join(OUTPUT_DIR, 'SubD1.png')
save_single_view(tmp1, out1, z_values_68, cmap_atrophy_name, -z_abs_max, z_abs_max)
sub_files.append(out1); tmp_files.append(tmp1)

# Figs 2-4: MIND fingerprints from the three seeds (blue-white-red)
for si, seed in enumerate(seeds):
    seed_idx = dk68_labels.index(seed)
    mind_seed = mind_matrix[seed_idx, :].copy()
    mind_max = mind_seed.max()
    mind_seed[seed_idx] = -mind_max
    tmp_m = os.path.join(OUTPUT_DIR, f'_tmp_mind_{si}.png')
    out_m = os.path.join(OUTPUT_DIR, f'SubD{2+si}.png')
    save_single_view(tmp_m, out_m, mind_seed, cmap_div_name, -mind_max, mind_max)
    sub_files.append(out_m); tmp_files.append(tmp_m)

# Fig 5: GOF scores (all views)
tmp5 = os.path.join(OUTPUT_DIR, '_tmp_5.png')
out5 = os.path.join(OUTPUT_DIR, 'SubD5.png')
save_single_view(tmp5, out5, gof_values_68, cmap_div_name, -gof_abs_max, gof_abs_max, keep_all_views=True)
sub_files.append(out5); tmp_files.append(tmp5)

titles = ['Z-score (Atrophy)', 'MIND from rh_cACC', 'MIND from rh_TemporalPole',
          'MIND from rh_Cuneus', 'GOF Score']
imgs = [plt.imread(f) for f in sub_files]


def find_content_range(v):
    gray = v.mean(axis=2) if v.ndim == 3 else v
    rows = np.where(gray.min(axis=1) < 0.98)[0]
    return (rows[0], rows[-1]) if len(rows) else (0, v.shape[0]-1)


ranges = [find_content_range(v) for v in imgs]
top, bottom = min(r[0] for r in ranges), max(r[1] for r in ranges)
imgs_cropped = [v[top:bottom+1, :, :3] for v in imgs]

# Align widths
single_w = max(imgs_cropped[i].shape[1] for i in range(4))
for i in range(4):
    if imgs_cropped[i].shape[1] < single_w:
        pad = np.ones((imgs_cropped[i].shape[0], single_w - imgs_cropped[i].shape[1], 3),
                      dtype=imgs_cropped[i].dtype)
        imgs_cropped[i] = np.concatenate([imgs_cropped[i], pad], axis=1)

H_GAP, V_GAP = 10, 30
gap_h = np.ones((imgs_cropped[0].shape[0], H_GAP, 3), dtype=imgs_cropped[0].dtype)
row1 = np.concatenate([imgs_cropped[0], gap_h, imgs_cropped[1], gap_h, imgs_cropped[2]], axis=1)
total_width = row1.shape[1]
max_subd5_w = total_width - single_w - H_GAP

subd5_raw = imgs_cropped[4]
h_raw, w_raw, _ = subd5_raw.shape
if w_raw > max_subd5_w:
    new_h = int(h_raw * (max_subd5_w / w_raw))
    subd5_img = Image.fromarray((subd5_raw * 255).astype(np.uint8))
    subd5_img = subd5_img.resize((max_subd5_w, new_h), Image.Resampling.LANCZOS)
    subd5_resized = np.array(subd5_img).astype(np.float32) / 255.0
    if new_h <= row1.shape[0]:
        pad_top = (row1.shape[0] - new_h) // 2
        pad_bottom = row1.shape[0] - new_h - pad_top
        subd5_final = np.ones((row1.shape[0], max_subd5_w, 3), dtype=subd5_resized.dtype)
        subd5_final[pad_top:pad_top+new_h, :, :] = subd5_resized
    else:
        subd5_final = subd5_resized[:row1.shape[0], :, :]
else:
    pad_w = max_subd5_w - w_raw
    pad_h_matrix = np.ones((subd5_raw.shape[0], pad_w, 3), dtype=subd5_raw.dtype)
    subd5_final = np.concatenate([subd5_raw, pad_h_matrix], axis=1)
    if subd5_final.shape[0] > row1.shape[0]:
        subd5_final = subd5_final[:row1.shape[0], :, :]

row2 = np.concatenate([imgs_cropped[3], gap_h, subd5_final], axis=1)
gap_v = np.ones((V_GAP, total_width, 3), dtype=row1.dtype)
combined = np.concatenate([row1, gap_v, row2], axis=0)
combined_u8 = (combined * 255).astype(np.uint8) if combined.max() <= 1.0 else combined.astype(np.uint8)
img_pil = Image.fromarray(combined_u8)

output_path = os.path.join(OUTPUT_DIR, "SubD.png")
img_pil.save(output_path)
print(f"\nComposite saved: {output_path}")
display(IPyImage(output_path))

for f in tmp_files:
    if os.path.exists(f):
        os.remove(f)
print(f"Region-list length: {len(dk68_labels)}")